# Cardiovascular Disease Prediction

Compare Decision Tree, Naive Bayes, KNN, and Neural Network classifiers on the cardiovascular dataset. This project is for education and research, not medical diagnosis.

## 1. Import packages and load the dataset

Use `data/cardio_train.csv` when running locally. In Colab, upload `cardio_train.csv` if the file is not found.

In [ ]:
from pathlib import Path
import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score, recall_score
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

DATA_PATH = Path("data/cardio_train.csv")
if not DATA_PATH.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = Path(next(iter(uploaded)))
    except ImportError as exc:
        raise FileNotFoundError("Put cardio_train.csv in the data folder or upload it in Colab.") from exc

df = pd.read_csv(DATA_PATH, sep=";")
df.columns = df.columns.str.strip().str.lower()
print(f"Dataset shape: {df.shape}")
print(df.head())

## 2. Clean and prepare features

Remove invalid target rows, convert age from days to years, mark implausible measurements as missing, and exclude the patient identifier.

In [ ]:
target = "cardio"
categorical_features = ["gender", "cholesterol", "gluc"]
numeric_features = ["age_years", "height", "weight", "ap_hi", "ap_lo", "smoke", "alco", "active"]
required = {"age", "gender", "height", "weight", "ap_hi", "ap_lo", "cholesterol", "gluc", "smoke", "alco", "active", target}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df[target] = pd.to_numeric(df[target], errors="coerce")
df = df[df[target].isin([0, 1])].copy()
for column in ["age", "gender", "height", "weight", "ap_hi", "ap_lo", "cholesterol", "gluc", "smoke", "alco", "active"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["age_years"] = df["age"] / 365.25
df.loc[~df["age_years"].between(18, 110), "age_years"] = np.nan
df.loc[~df["height"].between(100, 250), "height"] = np.nan
df.loc[~df["weight"].between(30, 300), "weight"] = np.nan
valid_bp = df["ap_hi"].between(60, 250) & df["ap_lo"].between(30, 200) & (df["ap_hi"] >= df["ap_lo"])
df.loc[~valid_bp, ["ap_hi", "ap_lo"]] = np.nan
for column in ["smoke", "alco", "active"]:
    df.loc[~df[column].isin([0, 1]), column] = np.nan
for column in categorical_features:
    valid_values = [1, 2] if column == "gender" else [1, 2, 3]
    df.loc[~df[column].isin(valid_values), column] = np.nan

X = df[numeric_features + categorical_features].copy()
y = df[target].astype(int)
print(f"Rows used: {len(df):,}")
print(f"Class counts:\n{y.value_counts().sort_index().rename(index={0: 'No disease', 1: 'Disease'})}")

## 3. Split the data and define the models

Imputation, one-hot encoding, and scaling are fitted inside each training fold to avoid data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

classifiers = {
    "Decision Tree": DecisionTreeClassifier(max_depth=12, min_samples_leaf=5, random_state=42),
    "Naive Bayes": GaussianNB(),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=7, weights="distance", n_jobs=-1),
    "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=150, n_iter_no_change=10, random_state=42)
}
models = {
    name: Pipeline([("preprocessor", preprocessor), ("classifier", classifier)])
    for name, classifier in classifiers.items()
}
print(f"Training rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print(f"Models: {', '.join(models)}")

## 4. Cross-validate and evaluate all classifiers

The best classifier is selected using mean cross-validation F1. Test metrics are reported separately.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
test_rows = []

for name, model in models.items():
    print(f"Training {name}...")
    cv_result = cross_validate(
        model, X_train, y_train, cv=cv,
        scoring={"accuracy": "accuracy", "f1": "f1"}, n_jobs=1
    )
    cv_rows.append({
        "model": name,
        "cv_accuracy_mean": cv_result["test_accuracy"].mean(),
        "cv_accuracy_std": cv_result["test_accuracy"].std(),
        "cv_f1_mean": cv_result["test_f1"].mean(),
        "cv_f1_std": cv_result["test_f1"].std()
    })
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()
    test_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1_score": f1_score(y_test, predictions, zero_division=0),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp
    })

cv_results = pd.DataFrame(cv_rows).sort_values("cv_f1_mean", ascending=False).reset_index(drop=True)
test_results = pd.DataFrame(test_rows)
best_name = cv_results.loc[0, "model"]
best_model = models[best_name]
print("\nCross-validation results, ranked by F1:")
print(cv_results.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("\nHeld-out test results:")
print(test_results.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print(f"\nSelected model: {best_name}")

## 5. Plot confusion matrices

Each confusion matrix uses the same held-out test set.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for axis, (name, model) in zip(axes.flat, models.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test, display_labels=["No disease", "Disease"],
        cmap="Blues", colorbar=False, ax=axis
    )
    axis.set_title(name)
fig.tight_layout()
plt.show()

## 6. Analyze feature importance

Permutation importance measures the change in F1 when each feature is shuffled. It describes model behavior, not causation.

In [ ]:
sample_size = min(2000, len(X_test))
sample_X = X_test.sample(n=sample_size, random_state=42)
sample_y = y_test.loc[sample_X.index]
importance_result = permutation_importance(
    best_model, sample_X, sample_y, scoring="f1",
    n_repeats=3, random_state=42, n_jobs=-1
)
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance_mean_f1_decrease": importance_result.importances_mean,
    "importance_std": importance_result.importances_std
}).sort_values("importance_mean_f1_decrease", ascending=False)
print(feature_importance.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

plot_data = feature_importance.sort_values("importance_mean_f1_decrease")
plt.figure(figsize=(9, 6))
plt.barh(plot_data["feature"], plot_data["importance_mean_f1_decrease"], xerr=plot_data["importance_std"], color="#3478a8")
plt.xlabel("Decrease in F1 when feature is shuffled")
plt.title(f"Feature importance for {best_name}")
plt.tight_layout()
plt.show()

## 7. Save results and the selected model

The saved pipeline includes preprocessing and can be loaded to classify new patient records with the same feature columns.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
cv_results.to_csv(output_dir / "cross_validation_metrics.csv", index=False)
test_results.to_csv(output_dir / "test_metrics.csv", index=False)
feature_importance.to_csv(output_dir / "feature_importance.csv", index=False)
joblib.dump(best_model, output_dir / "best_model.joblib")
summary = {
    "rows_used": int(len(df)),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "cross_validation_folds": 5,
    "selected_model": best_name,
    "selection_metric": "mean cross-validation F1-score",
    "best_model_test_metrics": test_results.set_index("model").loc[best_name].to_dict(),
    "top_features": feature_importance.head(5)["feature"].tolist(),
    "clinical_notice": "Educational research only; not a diagnostic tool."
}
with (output_dir / "summary.json").open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)
print(f"Saved metrics, feature importance, model, and summary to {output_dir.resolve()}")
print("This model is not a substitute for assessment by a qualified healthcare professional.")